# Chapter 5 — Fine-tuning a building detector on leaf-off imagery

ParcelVision detects residential building footprints for parcel-level property
intelligence. Our measured eval (`worker/scripts/eval_detectors.py`) surfaced a
concrete gap:

| imagery | detector | precision | recall | F1 |
|---------|----------|----------:|-------:|---:|
| leaf-off | RF-DETR (prod default) | 0.59 | 0.69 | **0.64** |
| leaf-off | Mask R-CNN (pretrained `building_footprints_usa`) | 0.18 | 0.38 | 0.25 |

The pretrained Mask R-CNN was trained on **leaf-ON** summer NAIP, so on the
**leaf-off** Missouri orthoimagery we actually use it over-detects badly. This
notebook closes that gap the honest way: **retrain on the imagery we deploy on**
— Missouri 6-inch leaf-off orthophotos labelled with Overture footprints — and
measure the before/after against the pretrained baseline.

> **Compute:** Mask R-CNN fine-tuning wants a GPU. On a CPU this runs but is
> slow; use a small `num_epochs`/`max_tiles` to rehearse, or run on Colab / a
> rented GPU for a real result. The headless, parameterized version of every
> step lives in `worker/scripts/finetune_buildings.py` (`--smoke` proves the
> pipeline end to end).

## 1. Setup

Run inside the worker image (has `geoai` + torch), or `pip install geoai-py`.
The helpers below reuse ParcelVision's own imagery fetch so training and
production share one leaf-off source.

In [ ]:
import sys
sys.path.insert(0, "../worker")  # ParcelVision worker package

import geoai
import geopandas as gpd
from pathlib import Path
from worker.pipeline.fetch import _fetch_mo_leafoff

# Training region: a residential swath of St. Louis County, kept clear of the
# eval AOI so we measure generalization, not memorization.
TRAIN_BBOX = [-90.3600, 38.6600, -90.3400, 38.6750]
TEST_BBOX  = [-90.3167, 38.6465, -90.3111, 38.6501]  # held-out residential AOI
WORK = Path("training_run"); WORK.mkdir(exist_ok=True)

## 2. Prepare training data

Fetch leaf-off imagery + Overture footprints (weak but authoritative labels),
then export aligned image/label tiles with `geoai.export_geotiff_tiles`. Overture
omits some garages/sheds, so labels are imperfect — good enough to teach the model
the leaf-off appearance of residential roofs.

In [ ]:
raster = _fetch_mo_leafoff(TRAIN_BBOX, WORK)[0]

geoai.download_overture_buildings(bbox=tuple(TRAIN_BBOX), output=str(WORK / "overture.geojson"))
gdf = gpd.read_file(WORK / "overture.geojson")
gdf = gdf[gdf.geometry.notna() & gdf.geometry.is_valid]
gdf["class"] = 1  # single foreground class
gdf[["class", "geometry"]].to_file(WORK / "labels.geojson")
print(len(gdf), "footprints")

tiles = geoai.export_geotiff_tiles(
    in_raster=str(raster),
    out_folder=str(WORK / "tiles"),
    in_class_data=str(WORK / "labels.geojson"),
    tile_size=512, stride=256, buffer_radius=0, skip_empty_tiles=True,
)

## 3. Fine-tune Mask R-CNN

Start from COCO-pretrained weights and fine-tune on the leaf-off tiles. Bump
`num_epochs` to ~100 on a GPU for a real result.

In [ ]:
geoai.train_MaskRCNN_model(
    images_dir=str(WORK / "tiles" / "images"),
    labels_dir=str(WORK / "tiles" / "labels"),
    output_dir=str(WORK / "models"),
    num_channels=3, pretrained=True,
    batch_size=4, num_epochs=100, learning_rate=0.005, val_split=0.2,
)

## 4. Before / after on a held-out leaf-off AOI

Load the fine-tuned checkpoint into the same `BuildingFootprintExtractor` the app
uses, run it on the held-out AOI, and score per-structure precision/recall/F1
against Overture — next to the pretrained baseline. Expect the fine-tuned model
to recover precision the pretrained one loses on leaf-off imagery.

In [ ]:
from geoai import BuildingFootprintExtractor
from worker.pipeline.postprocess import postprocess

def _score(pred, ref):
    """Per-structure precision/recall/F1 at IoU>=0.5 (greedy 1:1 match)."""
    if pred is None or pred.empty:
        return {"detections": 0, "precision": 0, "recall": 0, "f1": 0}
    pred = pred.to_crs(ref.crs); tp = 0; used = set()
    for pg in pred.geometry:
        pg = pg if pg.is_valid else pg.buffer(0)
        for ri, rg in enumerate(ref.geometry):
            if ri in used:
                continue
            inter = pg.intersection(rg).area
            if inter and inter / (pg.area + rg.area - inter) >= 0.5:
                used.add(ri); tp += 1; break
    p = tp / max(len(pred), 1); r = tp / max(len(ref), 1)
    return {"detections": len(pred), "precision": round(p, 3),
            "recall": round(r, 3), "f1": round(2 * p * r / max(p + r, 1e-9), 3)}

test_tif = _fetch_mo_leafoff(TEST_BBOX, WORK / "test")[0]
geoai.download_overture_buildings(bbox=tuple(TEST_BBOX), output=str(WORK / "test_overture.geojson"))
ref = gpd.read_file(WORK / "test_overture.geojson")
ref = ref.to_crs(ref.estimate_utm_crs())

def run(model_path):
    ex = BuildingFootprintExtractor(model_path=model_path) if model_path else BuildingFootprintExtractor()
    gdf = ex.process_raster(str(test_tif), confidence_threshold=0.5, min_object_area=50)
    return _score(postprocess(gdf, TEST_BBOX), ref)

ckpt = str(next((WORK / "models").glob("*.pth")))
print("pretrained :", run(None))
print("fine-tuned :", run(ckpt))

## Notes

- **Labels are weak.** Overture misses small outbuildings and lags new
  construction, which caps achievable F1 and slightly punishes a *correct*
  detection of an unlabelled shed. A hand-labelled test tile gives a truer read.
- **Next steps:** more training regions/vintages; try fine-tuning RF-DETR
  (`geoai.rfdetr_train`) since it is the production default; and exploit the
  native 0.15 m leaf-off resolution with a matched-resolution model.
- The trained `.pth` drops into production by pointing
  `BuildingFootprintExtractor(model_path=...)` at it — the `local_cpu` backend
  already accepts a custom checkpoint path.